In [ ]:
import equinox as eqx
import grain
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
from context_flux_no.data.sources import WellDatasetSourceBase, ZarrWellDatasetSource
from context_flux_no.models.multiphysics import (
    AbstractMultiphysicsOperator,
    DISCO,
    DPOT,
)
from context_flux_no.models.multiphysics.hyperfluxfno import (
    HyperFluxFNOLocal,
    HyperNeuralOperator,
    HyperNeuralOperatorv2
)
from context_flux_no.training.trainer import Trainer
from jaxtyping import Array, Float, PRNGKeyArray


jax.config.update("jax_default_device", jax.devices("gpu")[6])

In [ ]:
transforms = [grain.transforms.Batch(batch_size=64)]
source_train = ZarrWellDatasetSource(
    "../../data/datasets",
    "euler_2d",
    "train",
    window_size=21,
    preload_into_ram=True,
)
dataloader = grain.DataLoader(
    data_source=source_train,
    sampler=grain.samplers.IndexSampler(len(source_train), shuffle=True, seed=0),
    operations=transforms,
    worker_count=0,
)
source_valid = ZarrWellDatasetSource(
    "../../data/datasets",
    "euler_2d",
    "valid",
    window_size=21,
    preload_into_ram=True,
)
dataloader_valid = grain.DataLoader(
    data_source=source_valid,
    sampler=grain.samplers.IndexSampler(len(source_valid), shuffle=True, seed=1),
    operations=transforms,
    worker_count=0,
    # read_options=grain.ReadOptions(num_threads=32, prefetch_buffer_size=1000),
)

In [ ]:
dataiter = iter(dataloader)
batch = next(dataiter)

In [ ]:
(batch.dtype.itemsize * batch.size) / 1e9

In [ ]:
batch.shape

In [ ]:
sample_idx = 5
fig, axes = plt.subplots(
    2, 5, figsize=(8, 4), constrained_layout=True, sharex=True, sharey=True
)
for i, ax in enumerate(axes.flat):
    ax.imshow(batch[sample_idx, i * 2, 1])

In [ ]:
dpot = DPOT(
    num_spatial_dims=2,
    in_channels=5,
    out_channels=5,
    in_timesteps=10,
    grid_size=(64, 64),
    patch_size=(8, 8),
    embedding_dim=128,
    max_frequency_modes=(10, 10),
    fno_depth=10,
    num_blocks=8,
    hidden_dim_patch=512,
    hidden_dim_fno=512,
    hidden_dim_output=512,
    key=jax.random.key(0),
)
print(dpot.num_parameters() / 1e6)

In [ ]:
hyperfluxfno = HyperFluxFNOLocal(
    num_spatial_dims=2,
    in_channels=5,
    in_timesteps=10,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(64, 64),
        patch_size=(8, 8),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    depth=5,
    lift_dim=128,
    stencil_size=(8, 8),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxfno.num_parameters() / 1e6)

In [ ]:
disco = DISCO(
    num_spatial_dims=2,
    channels=5,
    embedding_dim=128,
    patch_size=8,
    num_hypernet_blocks=3,
    droppath=0.1,
    num_hypernet_heads=8,
    mlp_hidden_dim=32,
    boundary_condition="periodic",
    opnet_hidden_channels=8,
    opnet_norm_groups=4,
    rtol=1e-4,
    atol=1e-6,
    max_steps=16,
    key=jax.random.key(0),
)
print(disco.num_parameters() / 1e6)

In [ ]:
hyperfluxfno = HyperNeuralOperator(
    num_spatial_dims=2,
    in_channels=4,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(128, 128),
        patch_size=(8, 8),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="FluxNO",
    target_network_kwargs=dict(
        stencil_widths=(8, 8), lift_dim=128, hidden_dim=128, depth=5
    ),
    lift_dim=None,
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxfno.num_parameters() / 1e6)

In [ ]:
hyperfluxfno2 = HyperNeuralOperator(
    num_spatial_dims=2,
    in_channels=4,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(128, 128),
        patch_size=(8, 8),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="NDFluxNO",
    target_network_kwargs=dict(
        normal_stencil_halfwidths=(8, 8), transverse_stencil_halfwidth=6, lift_dim=128, hidden_dim=128, depth=4, stack_grid=False
    ),
    lift_dim=None,
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxfno2.num_parameters() / 1e6)

In [ ]:
hyperfluxfno2(jnp.ones((21, 4, 128, 128)), (0.01, 1 / 128, 1 / 128))

In [ ]:
hyperfluxfno3 = HyperNeuralOperator(
    num_spatial_dims=2,
    in_channels=4,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(128, 128),
        patch_size=(8, 8),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="ConvNextFluxNO",
    target_network_kwargs=dict(lift_dim=64, kernel_size=5, depth=4),
    lift_dim=None,
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxfno3.num_parameters() / 1e6)

In [ ]:
hyperfluxfno = HyperNeuralOperatorv2(
    num_spatial_dims=2,
    in_channels=4,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(128, 128),
        patch_size=(8, 8),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    normal_stencil_halfwidths=(4, 4),
    transverse_stencil_halfwidth=2,
    reconstructions_per_channel = 16,
    depth_target = 6,
    hidden_dim = 128,
    width_hyper=128,
    blocks_hyper=2,
    hypernet_init = "bias-hyperinit",
    key=jax.random.key(0),
)
print(hyperfluxfno.num_parameters() / 1e6)

In [ ]:
def get_loss_args(source: WellDatasetSourceBase):
    dt = source.metadata_common["temporal_resolution"]
    dxs = source.metadata_common["spatial_resolution"]
    return (dt, *dxs)

In [ ]:
args = get_loss_args(source_train)
compiled = (
    eqx.filter_jit(lambda model, x, args_, key: model(x, args_, key=key))
    .lower(hyperfluxfno, batch[0, :-1], args, jax.random.key(0))
    .compile()
)
flops = compiled.compiled.cost_analysis()["flops"] / 1e9
print(flops)

In [ ]:
eqx.filter_vmap(hyperfluxfno, in_axes=(0, None))(batch[:, :-1], args)[
    0
].block_until_ready()

In [ ]:
def loss_fn(
    model: AbstractMultiphysicsOperator,
    u: Float[Array, "batch time dim ..."],
    args,
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], dict]:
    u0, u1 = u[:, :-1], u[:, -1]
    keys = jax.random.split(key, u0.shape[0])
    u1_pred: Float[Array, "batch dim ..."] = eqx.filter_vmap(
        lambda u_, key_: model(u_, args, key=key_)
    )(u0, keys)[0]
    return jnp.mean((u1 - u1_pred) ** 2), dict()


trainer = Trainer(
    optax.adamw(1e-3),
    loss_fn,
    "./",
    None,
    {"entity": "jhko725", "project": "euler_test"},
    config_dict={"model": ""},
)

In [ ]:
trainer.train(
    hyperfluxfno3,
    dataloader,
    dataloader_valid,
    get_loss_args(source_train),
    num_steps=500,
    seed=0,
)